In [7]:
import pandas as pd
!pip install --upgrade gdown

import gdown
import os

os.makedirs("/kaggle/working/drive_files", exist_ok=True)


file_id = "1LiJkSCYlYwld-ssS_YgGvKc2fI09jSnp"
output_path = "/kaggle/working/drive_files/data.csv"

url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(id=file_id, output=output_path, quiet=False)
df = pd.read_csv(output_path)

df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 323.1 kB/s eta 0:00:000:00:01
  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


Downloading...
From: https://drive.google.com/uc?id=1LiJkSCYlYwld-ssS_YgGvKc2fI09jSnp
To: /kaggle/working/drive_files/data.csv
100%|██████████| 35.8k/35.8k [00:00<00:00, 34.3MB/s]


,file_id,image_path,label_path
0,0,/content/drive/MyDrive/satalite data/data/imag...,/content/drive/MyDrive/satalite data/data/labe...
1,1,/content/drive/MyDrive/satalite data/data/imag...,/content/drive/MyDrive/satalite data/data/labe...
2,3,/content/drive/MyDrive/satalite data/data/imag...,/content/drive/MyDrive/satalite data/data/labe...
3,8,/content/drive/MyDrive/satalite data/data/imag...,/content/drive/MyDrive/satalite data/data/labe...
4,5,/content/drive/MyDrive/satalite data/data/imag...,/content/drive/MyDrive/satalite data/data/labe...


In [3]:
df = pd.read_csv(output_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5013504 entries, 0 to 5013503
Data columns (total 7 columns):
 #   Column               Dtype  
---  ------               -----  
 0   NIR                  float64
 1   SWIR1                float64
 2   SWIR2                float64
 3   Merit DEM            float64
 4   Copernicus DEM       float64
 5   ESA world cover map  float64
 6   Water occ prob       float64
dtypes: float64(7)
memory usage: 267.8 MB


In [30]:
import numpy as np

def process_channels(df, channels, all_channel_names=None):

    if all_channel_names is None:
        raise ValueError("all_channel_names must be provided.")

    # Check requested channels
    channels = [ch for ch in channels if ch not in all_channel_names]


    df_selected = df[channels].copy()

    df_selected = df_selected.astype(np.float32)

    return df_selected

In [32]:
channels=['NIR', 'SWIR1', 'SWIR2', 'Merit DEM', 'Copernicus DEM', 'ESA world cover map', 'Water occ prob']
all_channels  = ["Coastal aerosol", "B", "G", "R", "NIR", "SWIR1", "SWIR2",
                "QA Band", "Merit DEM", "Copernicus DEM", "ESA world cover map", "Water occ prob"]

df= process_channels(df, channels =channels , all_channel_names = all_channels )

In [33]:
import os
import torch

def save_checkpoint(state, checkpoint_dir="checkpoints", filename="last_checkpoint.pth"):
    os.makedirs(checkpoint_dir, exist_ok=True)
    filepath = os.path.join(checkpoint_dir, filename)
    torch.save(state, filepath)
    print(f"--> Saved checkpoint to {filepath}")

def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None):
    if not os.path.exists(checkpoint_path):
        print("--> No checkpoint found, starting training from scratch.")
        return 0  

    print(f"--> Loading checkpoint from '{checkpoint_path}'...")
    checkpoint = torch.load(checkpoint_path)


    model.load_state_dict(checkpoint['state_dict'])
    if optimizer and 'optimizer' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer'])
    if scheduler and 'scheduler' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler'])

    start_epoch = checkpoint['epoch'] + 1
    print(f"--> Successfully loaded checkpoint. Resuming from Epoch {start_epoch}")
    return start_epoch

In [34]:
import albumentations as A
from albumentations.pytorch import ToTensorV2


train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),

    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(p=0.3),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.4),


    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Added padding=1 to preserve spatial dimensions (Same Padding)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):

    def __init__(self, in_channels=1, out_channels=1, num_classes=None):
        super().__init__()

        
        if num_classes is not None:
            out_channels = num_classes

        # <<(Encoder)>>
        self.down_conv1 = DoubleConv(in_channels, 64)
        self.down_conv2 = DoubleConv(64, 128)
        self.down_conv3 = DoubleConv(128, 256)
        self.down_conv4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        
        self.bottleneck = DoubleConv(512, 1024)

        # <<(Decoder)>>
        self.up_trans4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up_conv4 = DoubleConv(1024, 512)

        self.up_trans3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up_conv3 = DoubleConv(512, 256)

        self.up_trans2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up_conv2 = DoubleConv(256, 128)

        self.up_trans1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up_conv1 = DoubleConv(128, 64)

        # Final Output Layer
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
      
        x1 = self.down_conv1(x)
        x2 = self.down_conv2(self.pool(x1))
        x3 = self.down_conv3(self.pool(x2))
        x4 = self.down_conv4(self.pool(x3))

        
        x_bottle = self.bottleneck(self.pool(x4))

        #  (No cropping; padding=1)
        d4 = self.up_trans4(x_bottle)
        d4 = self.up_conv4(torch.cat([x4, d4], dim=1))

        d3 = self.up_trans3(d4)
        d3 = self.up_conv3(torch.cat([x3, d3], dim=1))

        d2 = self.up_trans2(d3)
        d2 = self.up_conv2(torch.cat([x2, d2], dim=1))

        d1 = self.up_trans1(d2)
        d1 = self.up_conv1(torch.cat([x1, d1], dim=1))

        out = self.final_conv(d1)
        return out

In [36]:
def dice_coeff(y_true, y_pred, smooth=1.0):
    intersection = tf.reduce_sum(y_true * y_pred, axis=-1)
    union = tf.reduce_sum(y_true, axis=-1) + tf.reduce_sum(y_pred, axis=-1)
    return (2.0 * intersection + smooth) / (union + smooth)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import rasterio



class SatellitePixelDataset(Dataset):

    def __init__(
        self, df, height, width,  water_threshold=0.5 ):

        self.df = df.reset_index(drop=True)

        self.height = height
        self.width = width

        
        self.water_threshold = water_threshold

        self.feature_cols = [
            'NIR',
            'SWIR1',
            'SWIR2',
            'Merit DEM',
            'Copernicus DEM',
            'ESA world cover map'
        ]

        self.target_col = 'Water occ prob'

       #num of pixels 

        expected_pixels = height * width

        if len(self.df) != expected_pixels:

            raise ValueError(
                f"DataFrame contains {len(self.df)} rows, "
                f"but height × width = {expected_pixels}.\n"
                f"Expected exactly one image/tile."
            )

    def __len__(self):

        
        return  len(self.df)

    def __getitem__(self, idx):

        channels = []

        for col in self.feature_cols:

            values = self.df[col].values

            image_channel = values.reshape(
                self.height,
                self.width
            ).astype(np.float32)


        image = np.stack(
            channels,
            axis=0
        )

       

        water = self.df[ self.target_col].values

        water = water.reshape( self.height, self.width ).astype(np.float32)

        water = np.nan_to_num(
            water,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )


        mask = ( water >= self.water_threshold ).astype(np.float32)

        # [H,W] → [1,H,W]

        mask = np.expand_dims(mask,  axis=0)


        image = torch.from_numpy(
            image
        ).float()

        mask = torch.from_numpy(
            mask
        ).float()

        return image, mask



train_df, val_df = train_test_split(df, test_size=0.3, random_state=42)
HEIGHT = 14
WIDTH = 14
train_dataset = SatellitePixelDataset(
    df=train_df ,
    height=HEIGHT,
    width=WIDTH,
    water_threshold=0.5
)

val_dataset = SatellitePixelDataset(
    df=val_df,
    height=HEIGHT,
    width=WIDTH ,
    water_threshold=0.5
)

# DataLoaders with defined batch sizes
train_loader = DataLoader(train_df, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_df, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_df)}, Validation samples: {len(val_df)}")

In [41]:
import os
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_dir = "/content/drive/MyDrive/satalite_checkpoints"
 
def fit(model, train_loader, val_loader, optimizer, criterion, epochs, PATH):
    checkpoint_dir = os.path.dirname(PATH) if PATH else "/content/drive/MyDrive/satalite_checkpoints"
   
    if checkpoint_dir:
        os.makedirs(checkpoint_dir, exist_ok=True)
        
    start_epoch = 0
    history = {'train_loss': [], 'val_loss': []}
    min_val_loss = float('inf')

  
    if PATH and os.path.exists(PATH):
        print(f"Loading checkpoint from file: {PATH}")
        checkpoint = torch.load(PATH, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        # Start at the next epoch
        start_epoch = checkpoint.get('epoch', 0) + 1
        min_val_loss = checkpoint.get('loss', float('inf'))
        print(f"Resuming training from epoch {start_epoch}")
    else:
        print("No checkpoint file found, starting training from scratch.")

    
    end_epoch = start_epoch + epochs
    for epoch in range(start_epoch, end_epoch):
        print(f"{epoch+1} trianing")
        train_loss = 0.0
        val_loss = 0.0
        
      
        model.train()
        for X, y in train_loader:
            X = X.float().to(device)
            y = y.float().to(device)
            X_patch, y_patch = prepare_batch_for_unet(X, y, target_size=(16, 16))

            optimizer.zero_grad()
            output = model(X_patch)
            loss = criterion(output, y_patch)
            loss.backward()
            optimizer.step()
            

            train_loss += loss.item()
            

        
        model.eval()
        with torch.no_grad():
            for X, y in val_loader:
                X = X.float().to(device)
                y = y.float().to(device)
                X_patch, y_patch = prepare_batch_for_unet(X, y, target_size=(16, 16))

                output = model(X_patch)
                loss = criterion(output, y_patch)
                val_loss += loss.item()
                
            

        #avg
        train_loss = train_loss / len(train_loader)
        val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch + 1:>2d}/{end_epoch:>2d}] - train_loss: {train_loss:.5f}, val_loss: {val_loss:.5f}")

        history['train_loss'].append(train_loss)
        
        history['val_loss'].append(val_loss)
      

       
        checkpoint_data = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': val_loss,
        }
        
        
        torch.save(checkpoint_data, PATH)

        
        if val_loss < min_val_loss:
            min_val_loss = val_loss
            best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
            torch.save(checkpoint_data, best_model_path)
            print(f"--> Saved new best model to {best_model_path} (val_loss: {val_loss:.5f})")

    return history, PATH

In [42]:
import torch
import torch.nn as nn

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, y_pred, y_true):
        #  sigmoid to convert logits to probabilities [0, 1]
        y_pred = torch.sigmoid(y_pred)

        # Flatten tensors across spatial dimensions
        y_pred = y_pred.view(-1)
        y_true = y_true.view(-1)

        intersection = (y_pred * y_true).sum()
        dice = (2.0 * intersection + self.smooth) / (y_pred.sum() + y_true.sum() + self.smooth)

        # Loss is 1 - Dice Coefficient
        return 1.0 - dice



In [ ]:
import torch.optim as optim
model = UNet(in_channels=6, out_channels=1)
model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
criterion = DiceLoss().to(device)
MODEL_PATH = os.path.join(checkpoint_dir, "UNET_model.pt")

history , last_checkpoint=fit(model,train_loader,val_loader,optimizer,criterion,epochs=5,PATH=MODEL_PATH)


In [ ]:
with plt.style.context(style='fivethirtyeight'):
    plt.figure(figsize=(18,5))
    plt.rcParams['font.size']=15
    plt.plot(list(range(1,11)),history['train_loss'],label='Train Loss',color='darkgray')
    plt.plot(list(range(1,11)),history['val_loss'],label='Validation Loss',color='lightgray')
    plt.title(label='Loss plot-graphs')
    plt.legend()
    plt.show()